### Links and Resources
| Method / Command | Description |
|------------------|-------------|
| [DeltaTable.update()](https://docs.delta.io/latest/api/python/spark/index.html) | Update rows matching a condition — Python API |
| [UPDATE (SQL)](https://docs.databricks.com/en/sql/language-manual/delta-update.html) | SQL syntax for updating rows in a Delta table |
| [DeltaTable.history()](https://docs.delta.io/latest/api/python/spark/index.html) | Returns the full transaction log — each UPDATE appears as a versioned entry |
| [operationMetrics](https://docs.databricks.com/en/delta/history.html#operation-metrics-keys) | Per-commit metrics: `numUpdatedRows`, `numCopiedRows`, `numAddedFiles`, etc. |
| [Time Travel](https://docs.databricks.com/en/delta/history.html) | Read earlier versions of a Delta table with `VERSION AS OF` or `TIMESTAMP AS OF` |
| [OPTIMIZE](https://docs.databricks.com/en/sql/language-manual/delta-optimize.html) | Compact the small files produced by incremental updates |
| [Deletion Vectors](https://docs.databricks.com/en/delta/deletion-vectors.html) | Soft-delete mechanism used by DELETE but NOT by UPDATE |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

# Updating Records with the Delta Lake API

`UPDATE` is a first-class operation in Delta Lake. Like `DELETE`, it is ACID-compliant, recorded in the transaction log, and supports time travel. Unlike `DELETE`, it cannot use **Deletion Vectors** — every UPDATE must physically rewrite the Parquet files that contain changed rows.

There are two equivalent syntaxes:

| Approach | API |
|----------|-----|
| Python (programmatic) | `DeltaTable.update(condition, set={"col": new_value})` |
| SQL | `UPDATE catalog.schema.table SET col = value WHERE condition` |

The `set` dictionary maps column names to new values. Values can be:
- A **SQL string literal** — `"1199.99"` or `"'discontinued'"`
- A **Column expression** — `F.col("price") * 0.9` or `F.lit(True)`
- A **SQL expression string** — `"price * 0.9"` or `"CASE WHEN ..."`

### Dataset used in this notebook

A fresh `product_inventory` table is created at the start. Each section performs one category of update and immediately shows the result. At the end all updates are reviewed together through `history()` and time travel.

## Setup — Create the Demo Table

In [ ]:
from pyspark.sql import Row, functions as F
from delta.tables import DeltaTable

TABLE = "population_metrics.default.product_inventory"

rows = [
    Row(product_id=1,  name="Laptop Pro 15",      category="Electronics", price=1299.99, in_stock=True),
    Row(product_id=2,  name="Wireless Mouse",      category="Electronics", price=25.49,  in_stock=True),
    Row(product_id=3,  name="USB-C Hub",           category="Electronics", price=39.99,  in_stock=False),
    Row(product_id=4,  name="Standing Desk",       category="Furniture",   price=499.00, in_stock=True),
    Row(product_id=5,  name="Ergonomic Chair",     category="Furniture",   price=349.00, in_stock=True),
    Row(product_id=6,  name="Monitor Arm",         category="Furniture",   price=89.95,  in_stock=False),
    Row(product_id=7,  name="Notebook (A5)",       category="Stationery",  price=4.99,   in_stock=True),
    Row(product_id=8,  name="Ballpoint Pen 10pk",  category="Stationery",  price=2.49,   in_stock=True),
    Row(product_id=9,  name="Sticky Notes 5pk",    category="Stationery",  price=3.79,   in_stock=True),
    Row(product_id=10, name="Whiteboard Marker",   category="Stationery",  price=5.99,   in_stock=False),
    Row(product_id=11, name="Coffee Mug",          category="Kitchen",     price=12.00,  in_stock=True),
    Row(product_id=12, name="Desk Lamp LED",       category="Electronics", price=45.00,  in_stock=True),
]

spark.createDataFrame(rows).write.format("delta").mode("overwrite").saveAsTable(TABLE)
print(f"Table '{TABLE}' created — version 0, 12 rows.")
display(spark.table(TABLE).orderBy("product_id"))

# 1. Update a Single Row — Python API (SQL String Condition)

`DeltaTable.update(condition, set)` is the Python entry point.

- `condition` — a SQL string or Spark Column that identifies which rows to update.
- `set` — a `dict` mapping column names (strings) to new values (SQL strings or Column expressions).

**Scenario:** The "Laptop Pro 15" has a new price of 1,199.99 after a promotional markdown.

In [ ]:
dt = DeltaTable.forName(spark, TABLE)

# Update a single row by primary key — SQL string condition, SQL string value
dt.update(
    condition="product_id = 1",
    set={"price": "1199.99"}
)

print("After updating Laptop Pro 15 price to 1199.99:")
display(spark.table(TABLE).orderBy("product_id"))

# 2. Update Multiple Rows — Python API (Column Expression)

The condition and the new values can both be **Spark Column expressions** instead of raw SQL strings.  
Column expressions are type-safe and composable with the rest of the PySpark API.

**Scenario:** All out-of-stock products (`in_stock = False`) have been restocked.

In [ ]:
dt = DeltaTable.forName(spark, TABLE)

# Condition: Column expression.  Value: F.lit() wraps a Python literal as a Column.
dt.update(
    condition=F.col("in_stock") == False,
    set={"in_stock": F.lit(True)}
)

print("After marking all out-of-stock products as in_stock = True:")
display(spark.table(TABLE).orderBy("product_id"))
# products 3, 6, 10 should now show in_stock = true

# 3. Update with a Computed Value — Arithmetic on Existing Columns

The `set` value can reference other columns in the same row using `F.col()`.  
This is how you express "add 10% to the current price" without knowing the current value upfront.

**Scenario:** Apply a 10% price increase to all Furniture items.

In [ ]:
dt = DeltaTable.forName(spark, TABLE)

# Computed value: multiply the existing price column by 1.10
dt.update(
    condition=F.col("category") == "Furniture",
    set={"price": F.round(F.col("price") * 1.10, 2)}
)

print("After 10% price increase on Furniture:")
display(
    spark.table(TABLE)
    .filter("category = 'Furniture'")
    .orderBy("product_id")
)
# Standing Desk: 499.00 → 548.90
# Ergonomic Chair: 349.00 → 383.90
# Monitor Arm:    89.95 → 98.95

# 4. Update Multiple Columns at Once

The `set` dictionary can contain as many key-value pairs as needed — all columns are updated atomically in a single commit.

**Scenario:** The "USB-C Hub" (product_id = 3) is being renamed and its category is being reclassified.

In [ ]:
dt = DeltaTable.forName(spark, TABLE)

# Update two columns in the same commit
dt.update(
    condition="product_id = 3",
    set={
        "name":     "'USB-C Hub Pro'",      # SQL string literal needs surrounding quotes
        "category": "'Accessories'",
        "price":    "49.99"
    }
)

print("After updating product_id = 3 (name, category, price):")
display(spark.table(TABLE).filter("product_id = 3"))

# 5. SQL UPDATE Syntax

`UPDATE table SET col = value [, col = value ...] WHERE condition` is the ANSI SQL form.  
It is fully equivalent to the Python API — both produce the same transaction log entry.

**Scenario:** The Coffee Mug price needs a correction via SQL.

In [ ]:
# SQL UPDATE — single column, single row
spark.sql(f"""
    UPDATE {TABLE}
    SET    price = 14.99
    WHERE  product_id = 11
""")

print("After SQL UPDATE on Coffee Mug (product_id = 11):")
display(spark.table(TABLE).filter("product_id = 11"))

# 6. SQL UPDATE with CASE WHEN — Conditional Multi-Value Update

`CASE WHEN` inside a `SET` clause lets you assign different values to the same column depending on the row's data.  
This is a single-pass operation — it touches each qualifying row only once.

**Scenario:** Apply a tiered discount — 15% off Electronics, 10% off Furniture, 5% off everything else.

In [ ]:
spark.sql(f"""
    UPDATE {TABLE}
    SET price = ROUND(
        CASE
            WHEN category = 'Electronics' THEN price * 0.85
            WHEN category = 'Furniture'   THEN price * 0.90
            ELSE                               price * 0.95
        END,
    2)
""")

print("After tiered CASE WHEN discount update:")
display(spark.table(TABLE).orderBy("category", "product_id"))

# 7. Update All Rows — No Condition

Omitting the `condition` parameter (Python) or the `WHERE` clause (SQL) applies the update to every row in the table.  
This is equivalent to a full-table transform and rewrites all Parquet files.

**Scenario:** Add a `last_updated` audit timestamp to every row using a SQL expression.

In [ ]:
# First, add the column (ALTER TABLE) so it exists in the schema
spark.sql(f"ALTER TABLE {TABLE} ADD COLUMN last_updated TIMESTAMP")

dt = DeltaTable.forName(spark, TABLE)

# No condition → update every row
dt.update(
    set={"last_updated": F.current_timestamp()}
)

print("After stamping every row with current_timestamp():")
display(spark.table(TABLE).orderBy("product_id"))

# 8. Audit Trail — History After All Updates

Every `update()` call or SQL `UPDATE` writes a new commit to `_delta_log/`.  
`history()` records the predicate (`operationParameters`) and key metrics (`operationMetrics`) for each commit.

In [ ]:
dt = DeltaTable.forName(spark, TABLE)

# Full history ordered from oldest to newest
display(
    dt.history()
    .select("version", "timestamp", "operation", "operationParameters", "operationMetrics")
    .orderBy("version")
)

# 9. Understanding `operationMetrics` for UPDATE

The `operationMetrics` column on an `UPDATE` entry contains several important counters.
Understanding them explains the I/O cost of row-level updates in columnar storage.

| Metric | Meaning |
|--------|---------|
| `numUpdatedRows` | Rows whose values actually changed |
| `numCopiedRows` | Rows that were in the same Parquet file as updated rows but did NOT themselves change — they had to be rewritten anyway because Parquet is immutable |
| `numAddedFiles` | New Parquet files written to replace the affected old files |
| `numRemovedFiles` | Old Parquet files replaced (still on storage until `VACUUM`) |

### The `numCopiedRows` cost — why UPDATE is more expensive than DELETE

With Deletion Vectors, a `DELETE` of one row writes a small bitmap and leaves the Parquet file untouched.  
An `UPDATE` **cannot** use Deletion Vectors because the actual cell values must change — this requires rewriting the Parquet file.

```
Parquet file: [row0, row1, row2, row3, row4]   (5 rows, 1 target row)

DELETE row2  →  Deletion Vector marks row2  (only ~bytes written)
UPDATE row2  →  New Parquet file: [row0, row1, row2_new, row3, row4]
                numUpdatedRows = 1,  numCopiedRows = 4
```

**`numCopiedRows` is the hidden cost of UPDATE** — the more rows share a file with the updated row, the higher the copy overhead.

In [ ]:
dt = DeltaTable.forName(spark, TABLE)

# Extract and display UPDATE-specific metrics for every UPDATE version
history_df = (
    dt.history()
    .filter("operation = 'UPDATE'")
    .select(
        "version",
        "timestamp",
        "operationParameters",
        F.col("operationMetrics")["numUpdatedRows"].alias("updatedRows"),
        F.col("operationMetrics")["numCopiedRows"].alias("copiedRows"),
        F.col("operationMetrics")["numAddedFiles"].alias("addedFiles"),
        F.col("operationMetrics")["numRemovedFiles"].alias("removedFiles"),
    )
    .orderBy("version")
)

display(history_df)

### Interpreting the metrics

Look at the ratio of `copiedRows` to `updatedRows`:

- **Single-row update** (product_id = 1) — `updatedRows = 1`, but `copiedRows` may be 11 if all 12 rows landed in one file. The update of 1 row forced a full file rewrite.
- **Category update** (Furniture, 3 rows) — `updatedRows = 3`, `copiedRows` = the other rows in the same file(s).
- **Full table update** (all rows, `last_updated`) — `copiedRows = 0` because every row was updated; there were no innocent bystanders to copy.

This ratio highlights when OPTIMIZE matters most: **many small targeted updates over time leave many small files**, each with high copy overhead on the next update. Periodic `OPTIMIZE` compacts these files and reduces future copy costs.

# 10. Time Travel After Updates

Because UPDATE rewrites Parquet files (instead of using Deletion Vectors), the **original files are preserved as orphans** until `VACUUM` removes them.  
This means time travel works identically for updates and deletes — you can read any prior version as long as its files are still within the retention window.

**Scenario:** Read the table at version 0 (before any updates) and compare to the current state.

In [ ]:
# Current state
print("Current table (all updates applied):")
display(spark.table(TABLE).orderBy("product_id"))

In [ ]:
# Version 0 — the original state before any update
df_v0 = spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF 0 ORDER BY product_id")
print("Version 0 (original — before any updates):")
display(df_v0)

In [ ]:
# Version 1 — after the single Laptop price update
df_v1 = spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF 1 ORDER BY product_id")
print("Version 1 (after Laptop price markdown to 1199.99):")
display(df_v1.filter("product_id = 1"))

### Recover from an Accidental Update Using Time Travel

Time travel is the safety net for bad updates — read the pre-update version into a DataFrame and overwrite the table.

In [ ]:
# Simulate an accidental update — zero out all prices
spark.sql(f"UPDATE {TABLE} SET price = 0")
print("After accidental UPDATE (prices set to 0):")
display(spark.table(TABLE).orderBy("product_id").select("product_id", "name", "price"))

In [ ]:
# Determine which version to restore from
dt = DeltaTable.forName(spark, TABLE)
display(
    dt.history()
    .select("version", "timestamp", "operation")
    .orderBy("version")
)

In [ ]:
# Identify the last good version (the one just before the accidental UPDATE)
# Then restore it with RESTORE — a single SQL command that reverts the table in-place

# First, find the accidental update version
last_version = dt.history(1).collect()[0]["version"]
good_version = last_version - 1     # the version just before the accident

print(f"Current (broken) version : {last_version}")
print(f"Restoring to version     : {good_version}")

spark.sql(f"RESTORE TABLE {TABLE} TO VERSION AS OF {good_version}")

print("\nTable after RESTORE:")
display(spark.table(TABLE).orderBy("product_id").select("product_id", "name", "price"))

### RESTORE is itself a commit

`RESTORE TABLE ... TO VERSION AS OF n` adds a new entry to the transaction log — it does not rewind the log.  
The accidental update version is still visible in `history()`, which preserves the full audit trail.

In [ ]:
dt = DeltaTable.forName(spark, TABLE)

display(
    dt.history()
    .select("version", "timestamp", "operation", "operationParameters")
    .orderBy("version")
)
# The RESTORE appears as its own version — the bad update is still in the log

# 11. OPTIMIZE After Updates

Every UPDATE produces new Parquet files for the affected rows. Repeated incremental updates over time lead to **file fragmentation** — many small files instead of a few large ones.  

### Why fragmentation hurts

Each Parquet file has a fixed overhead: Spark opens it, reads its footer to get column statistics, and then reads the data pages. With hundreds of tiny files, this overhead dominates the actual data read time.

### What OPTIMIZE does after updates

| Before OPTIMIZE | After OPTIMIZE |
|-----------------|----------------|
| Many small files (one per UPDATE batch) | Few large files (typically ~1 GB each) |
| High per-file open overhead | Low per-file open overhead |
| Sparse column statistics | Dense column statistics — better data skipping |
| Old pre-update files still on storage | Old files become orphans (removed later by VACUUM) |

In [ ]:
# Storage state before OPTIMIZE — shows file count after many updates
dt = DeltaTable.forName(spark, TABLE)
detail_before = dt.detail().select("numFiles", "sizeInBytes").collect()[0]
print("Before OPTIMIZE:")
print(f"  numFiles   : {detail_before['numFiles']}")
print(f"  sizeInBytes: {detail_before['sizeInBytes']}")

In [ ]:
# Run OPTIMIZE — compacts fragmented files
optimize_result = spark.sql(f"OPTIMIZE {TABLE}")
display(optimize_result)    # returns a summary of files added / removed

dt = DeltaTable.forName(spark, TABLE)
detail_after = dt.detail().select("numFiles", "sizeInBytes").collect()[0]
print("\nAfter OPTIMIZE:")
print(f"  numFiles   : {detail_after['numFiles']}")
print(f"  sizeInBytes: {detail_after['sizeInBytes']}")

In [ ]:
# Inspect the OPTIMIZE history entry
dt = DeltaTable.forName(spark, TABLE)
display(
    dt.history()
    .filter("operation = 'OPTIMIZE'")
    .select(
        "version", "timestamp", "operationParameters",
        F.col("operationMetrics")["numFilesAdded"].alias("filesAdded"),
        F.col("operationMetrics")["numFilesRemoved"].alias("filesRemoved"),
        F.col("operationMetrics")["numDeletionVectorsRemoved"].alias("dvsRemoved"),
    )
    .orderBy("version")
)

# Summary

## `DeltaTable.update()` — Syntax Reference

```python
# SQL string condition + SQL string value
dt.update("product_id = 1", {"price": "1199.99"})

# Column expression condition + Column expression value
dt.update(F.col("in_stock") == False, {"in_stock": F.lit(True)})

# Computed value referencing another column
dt.update("category = 'Furniture'", {"price": F.round(F.col("price") * 1.10, 2)})

# Multiple columns in one commit
dt.update("product_id = 3", {"name": "'USB-C Hub Pro'", "price": "49.99"})

# All rows — no condition
dt.update(set={"last_updated": F.current_timestamp()})
```

## UPDATE vs DELETE — Internal Mechanics

| | DELETE | UPDATE |
|-|--------|--------|
| **Can use Deletion Vectors?** | Yes — only a bitmap written | No — values change, file must be rewritten |
| **Parquet files rewritten?** | No (with DVs enabled) | Yes — always |
| **I/O cost for 1 row** | Tiny bitmap file | Rewrite of the whole containing file |
| **`numCopiedRows`** | 0 (no file rewrite) | Non-zero — other rows in the same file |
| **Old files become orphans?** | No (DV, file untouched) | Yes — removed by VACUUM later |

## Key `operationMetrics` for UPDATE

| Metric | What it tells you |
|--------|-------------------|
| `numUpdatedRows` | Rows actually changed |
| `numCopiedRows` | Rows rewritten as collateral — hidden cost of update |
| `numAddedFiles` | New Parquet files created |
| `numRemovedFiles` | Old Parquet files replaced (still on storage until VACUUM) |

## Lifecycle: Update → History → OPTIMIZE → VACUUM

```
UPDATE rows
  └─► Affected Parquet files are rewritten (old files become orphans)
       └─► history() records version, predicate, numUpdatedRows, numCopiedRows
            └─► OPTIMIZE
                  └─► Compacts fragmented files from repeated updates
                       └─► VACUUM (after retention period)
                             └─► Orphan files permanently removed from storage
```

## Accidental Update Recovery

```sql
-- Identify the last good version
DESCRIBE HISTORY table_name;

-- Restore in-place (adds a new commit — audit trail preserved)
RESTORE TABLE table_name TO VERSION AS OF <good_version>;
```